In [1]:
import csv
import json
import time
import requests

# ==============================================================================
# CONFIGURATION SCRAPERAPI & CIBLES
# ==============================================================================
# 1. REMPLACEZ PAR VOTRE CLÉ SCRAPERAPI :
API_KEY = "7f89125c440bc88d18a73169651699d0"

SCRAPER_API_URL = "http://api.scraperapi.com"

# Headers pour simuler un navigateur standard
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://www.sofascore.com",
    "Referer": "https://www.sofascore.com/"
}

# ==============================================================================
# DICTIONNAIRES DES CLUBS PAR LIGUE (IDs Sofascore à jour)
# ==============================================================================

LEAGUES = {
    "la_liga": {
        "real_madrid": 2829,
        "barcelona": 2817,
        "atletico_madrid": 2836,
        "girona": 24264,
        "real_sociedad": 2824,
        "athletic_bilbao": 2825,
        "real_betis": 2816,
        "villarreal": 2819,
        "valencia": 2828,
        "sevilla": 2833,
        "osasuna": 2820,
        "getafe": 2859,
        "celta_vigo": 2821,
        "mallorca": 2823,
        "rayo_vallecano": 2818,
        "las_palmas": 2826,
        "alaves": 2885,
        "leganes": 2845,
        "real_valladolid": 2831,
        "espanyol": 2814
    },
    "bundesliga": {
        "bayern_munich": 2672,
        "bayer_leverkusen": 2681,
        "vfb_stuttgart": 2677,
        "borussia_dortmund": 2673,
        "rb_leipzig": 23826,
        "eintracht_frankfurt": 2674,
        "hoffenheim": 2569,
        "heidenheim": 5802,
        "werder_bremen": 2675,
        "freiburg": 2538,
        "augsburg": 2700,
        "wolfsburg": 2690,
        "mainz_05": 2679,
        "borussia_monchengladbach": 2670,
        "union_berlin": 2547,
        "vfl_bochum": 2676,
        "st_pauli": 2521,
        "holstein_kiel": 5313
    },
    "serie_a": {
        "inter": 2697,
        "ac_milan": 2692,
        "juventus": 2687,
        "atalanta": 2685,
        "bologna": 2688,
        "roma": 2702,
        "lazio": 2699,
        "fiorentina": 2693,
        "torino": 2696,
        "napoli": 2714,
        "genoa": 2713,
        "monza": 2729,
        "hellas_verona": 2701,
        "lecce": 2689,
        "udinese": 2695,
        "cagliari": 2719,
        "empoli": 2703,
        "parma": 2691,
        "como": 2705,
        "venezia": 2698
    },
    "premier_league": {
        "arsenal": 42,
        "aston_villa": 40,
        "bournemouth": 60,
        "brentford": 50,
        "brighton": 30,
        "chelsea": 38,
        "crystal_palace": 33,
        "everton": 48,
        "fulham": 43,
        "ipswich_town": 11,
        "leicester_city": 14,
        "liverpool": 44,
        "manchester_city": 17,
        "manchester_united": 35,
        "newcastle_united": 39,
        "nottingham_forest": 15,
        "southampton": 45,
        "tottenham_hotspur": 33,
        "west_ham_united": 37,
        "wolverhampton_wanderers": 3
    }
}

# ==============================================================================
# REQUÊTE VIA SCRAPERAPI
# ==============================================================================

def get_json_via_scraperapi(target_url):
    payload = {
        "api_key": API_KEY,
        "url": target_url,
        "keep_headers": "true"
    }
    try:
        print(f"Requête API en cours pour : {target_url}...")
        response = requests.get(SCRAPER_API_URL, params=payload, headers=HEADERS, timeout=60)

        if response.status_code == 200:
            return response.json()
        else:
            print(f"[Erreur API {response.status_code}] Impossible de récupérer les données.")
            return None
    except Exception as e:
        print(f"[Exception] Erreur lors de la requête : {e}")
        return None

# ==============================================================================
# PIPELINE DE SCRAPING DE JOUEURS
# ==============================================================================

def scrape_club_players(league_name, club_name, club_id):
    target_url = f"https://api.sofascore.com/api/v1/team/{club_id}/players"
    output_file = f"sofascore_players_{league_name}_{club_name}.csv"
    
    print(f"\n--- SCRAPING COMMENCÉ POUR : {club_name.upper()} (Ligue: {league_name.upper()}, ID: {club_id}) ---")

    data = get_json_via_scraperapi(target_url)
    if not data or "players" not in data:
        print(f"[Échec] Aucun joueur récupéré pour {club_name}. L'ID est peut-être incorrect ou l'API a bloqué la requête.")
        return

    players_list = data["players"]
    print(f"[Succès] {len(players_list)} joueurs trouvés dans l'effectif de {club_name.capitalize()}.")

    # AJOUT DES COLONNES "league" ET "club" DANS LES HEADERS
    csv_headers = [
        "player_id",
        "name",
        "short_name",
        "position",
        "jersey_number",
        "age",
        "country",
        "market_value_raw",
        "league",
        "club"
    ]

    with open(output_file, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=csv_headers)
        writer.writeheader()

        for item in players_list:
            player_info = item.get("player", {})
            
            player_id = player_info.get("id")
            name = player_info.get("name")
            short_name = player_info.get("shortName")
            position = player_info.get("position", "Inconnu")
            jersey_number = player_info.get("jerseyNumber", "-")
            
            dob_timestamp = player_info.get("dateOfBirthTimestamp")
            age = None
            if dob_timestamp:
                age = int((time.time() - dob_timestamp) / (365.25 * 24 * 3600))

            country = player_info.get("country", {}).get("name", "Inconnu")
            market_value_raw = player_info.get("proposedMarketValue", "Non renseigné")

            # ÉCRITURE AVEC LES NOUVELLES DONNÉES DE LIGUE ET DE CLUB
            writer.writerow({
                "player_id": player_id,
                "name": name,
                "short_name": short_name,
                "position": position,
                "jersey_number": jersey_number,
                "age": age,
                "country": country,
                "market_value_raw": market_value_raw,
                "league": league_name,
                "club": club_name
            })

    print(f"[SUCCÈS] Données exportées avec succès dans '{output_file}' !")

def main():
    # Vous pouvez ajouter ou retirer "premier_league" de cette liste au besoin
    ligues_a_scraper = ["la_liga", "bundesliga", "serie_a", "premier_league"] 
    
    print("=== DÉMARRAGE DU SCRAPING GLOBAL ===")
    
    for league in ligues_a_scraper:
        print(f"\n==========================================")
        print(f" DEBUT DU SCRAPING DE LA LIGUE : {league.upper()}")
        print(f"==========================================")
        
        clubs_dict = LEAGUES[league]
        for club_name, club_id in clubs_dict.items():
            scrape_club_players(league, club_name, club_id)
            print("Pause de 2 secondes avant le prochain club...")
            time.sleep(2)
        
    print("\n=== TOUT LE SCRAPING EST TERMINÉ AVEC SUCCÈS ! ===")

if __name__ == "__main__":
    main()

=== DÉMARRAGE DU SCRAPING GLOBAL ===

 DEBUT DU SCRAPING DE LA LIGUE : LA_LIGA

--- SCRAPING COMMENCÉ POUR : REAL_MADRID (Ligue: LA_LIGA, ID: 2829) ---
Requête API en cours pour : https://api.sofascore.com/api/v1/team/2829/players...


[Succès] 30 joueurs trouvés dans l'effectif de Real_madrid.
[SUCCÈS] Données exportées avec succès dans 'sofascore_players_la_liga_real_madrid.csv' !
Pause de 2 secondes avant le prochain club...



--- SCRAPING COMMENCÉ POUR : BARCELONA (Ligue: LA_LIGA, ID: 2817) ---
Requête API en cours pour : https://api.sofascore.com/api/v1/team/2817/players...


[Succès] 28 joueurs trouvés dans l'effectif de Barcelona.
[SUCCÈS] Données exportées avec succès dans 'sofascore_players_la_liga_barcelona.csv' !
Pause de 2 secondes avant le prochain club...



--- SCRAPING COMMENCÉ POUR : ATLETICO_MADRID (Ligue: LA_LIGA, ID: 2836) ---
Requête API en cours pour : https://api.sofascore.com/api/v1/team/2836/players...


...


--- SCRAPING COMMENCÉ POUR : WEST_HAM_UNITED (Ligue: PREMIER_LEAGUE, ID: 37) ---
Requête API en cours pour : https://api.sofascore.com/api/v1/team/37/players...


[Succès] 33 joueurs trouvés dans l'effectif de West_ham_united.
[SUCCÈS] Données exportées avec succès dans 'sofascore_players_premier_league_west_ham_united.csv' !
Pause de 2 secondes avant le prochain club...



--- SCRAPING COMMENCÉ POUR : WOLVERHAMPTON_WANDERERS (Ligue: PREMIER_LEAGUE, ID: 3) ---
Requête API en cours pour : https://api.sofascore.com/api/v1/team/3/players...


[Succès] 32 joueurs trouvés dans l'effectif de Wolverhampton_wanderers.
[SUCCÈS] Données exportées avec succès dans 'sofascore_players_premier_league_wolverhampton_wanderers.csv' !
Pause de 2 secondes avant le prochain club...



=== TOUT LE SCRAPING EST TERMINÉ AVEC SUCCÈS ! ===


In [2]:
import os
import glob
import pandas as pd

def fusionner_fichiers_csv():
    print("--- DÉMARRAGE DE LA FUSION DES FICHIERS ---")
    
    # 1. Trouver tous les fichiers CSV générés par vos scrapers
    # Ici, on cherche tous les fichiers commençant par "sofascore_" dans le dossier courant
    fichiers_csv = glob.glob("sofascore_players_*.csv")
    
    if not fichiers_csv:
        print("[Échec] Aucun fichier CSV commençant par 'sofascore_' n'a été trouvé dans ce dossier.")
        return

    print(f"Fichiers trouvés ({len(fichiers_csv)}) :")
    for f in fichiers_csv:
        print(f"  - {f}")

    liste_dataframes = []

    # 2. Lecture de chaque fichier avec Pandas
    for fichier in fichiers_csv:
        try:
            # L'encodage 'utf-8-sig' permet de gérer correctement les accents espagnols/français et l'import Excel
            df = pd.read_csv(fichier, encoding="utf-8-sig")
            liste_dataframes.append(df)
        except Exception as e:
            print(f"  [Erreur] Impossible de lire le fichier {fichier} : {e}")

    if not liste_dataframes:
        print("[Échec] Aucun fichier n'a pu être lu correctement.")
        return

    # 3. Fusion de tous les DataFrames
    # "outer" permet de garder toutes les colonnes, même si un fichier a un champ légèrement différent
    df_final = pd.concat(liste_dataframes, ignore_index=True, join="outer")

    # 4. Sauvegarde dans le fichier final
    output_filename = "sofascore_database_globale.csv"
    df_final.to_csv(output_filename, index=False, encoding="utf-8-sig")

    print(f"\n[SUCCÈS] {len(df_final)} lignes cumulées fusionnées avec succès !")
    print(f"Fichier final disponible sous le nom : '{output_filename}'")

if __name__ == "__main__":
    fusionner_fichiers_csv()

--- DÉMARRAGE DE LA FUSION DES FICHIERS ---
Fichiers trouvés (71) :
  - sofascore_players_bundesliga_augsburg.csv
  - sofascore_players_bundesliga_bayern_munich.csv
  - sofascore_players_bundesliga_bayer_leverkusen.csv
  - sofascore_players_bundesliga_borussia_dortmund.csv
  - sofascore_players_bundesliga_eintracht_frankfurt.csv
  - sofascore_players_bundesliga_freiburg.csv
  - sofascore_players_bundesliga_hoffenheim.csv
  - sofascore_players_bundesliga_mainz_05.csv
  - sofascore_players_bundesliga_rb_leipzig.csv
  - sofascore_players_bundesliga_union_berlin.csv
  - sofascore_players_bundesliga_vfb_stuttgart.csv
  - sofascore_players_bundesliga_vfl_bochum.csv
  - sofascore_players_bundesliga_werder_bremen.csv
  - sofascore_players_bundesliga_wolfsburg.csv
  - sofascore_players_la_liga_alaves.csv
  - sofascore_players_la_liga_athletic_bilbao.csv
  - sofascore_players_la_liga_atletico_madrid.csv
  - sofascore_players_la_liga_barcelona.csv
  - sofascore_players_la_liga_celta_vigo.csv
  - 


[SUCCÈS] 2236 lignes cumulées fusionnées avec succès !
Fichier final disponible sous le nom : 'sofascore_database_globale.csv'
